---
### **준비**

In [1]:
%%html
<style>
    body {
        /* change --vscode-font-family if it's different on your system */
        --vscode-font-family: "SeoulNamsanC";
    }
</style>

In [1]:
import os
import tensorflow as tf
import numpy as np
from pathlib import Path
import numpy as np
import keras

2026-03-15 21:37:27.056976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# TensorFlow 로그 레벨 설정 (0: 모든 로그, 1: 경고 및 오류, 2: 오류만, 3: 로그 없음)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
# 전체 파일에서 랜덤 시드 고정
tf.random.set_seed(42)
np.random.seed(42)

In [3]:
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ memory_growth enabled")
    except Exception as e:
        print("memory_growth set failed:", e)

2026-03-15 21:37:58.112819: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


---
## **목차 (색인)**

**3부는 ~~~**

- **16장 전체 커버 내용: RNN과 어텐션을 사용한 자연어 처리**
    - 문자 단위 RNN: 문장에서 다음 글자 예측
        - 상태가 없는 RNN (stateless)
        - 상태가 있는 RNN (stateful)
    - 단어 시퀀스 단위 문장 처리
        - 감성 분석 (IMDB 리뷰) **// 1부**
        - 기계 번역: 인코더-디코더 구조 -> 양방향 RNN, 빔서치 실습 **// 2부**
    - **Attention 메커니즘**
        - **RNN 기반 인코더-디코더 성능 향상**
        - **트랜스포머(Transformer) 구조 소개**
    - **최신 NLP 모델 및 라이브러리 소개 (GPT, BERT, Hugging Face Transformers) // 3부**
- 스터디 완료 후 추가 예정

---
# **16. RNN과 어텐션을 사용한 자연어 처리**
---

- 커버 내용 및 순서
    - **문자 단위 RNN : 문장에서 다음 글자를 예측**
        1. 상태가 없는 RNN(stateless) : 각 반복에서 랜덤하게 택한 텍스트의 일부로 학습, 나머지 텍스트 사용 X
        2. 상태가 있는 RNN(statefull) : 은닉 상태를 유지하고 중지된 시점부터 이어서 상태를 반영, 더 긴 패턴 가능
    - **문자가 아니라 단어의 시퀀스 단위로 문장을 다룸**
        1. 감성 분석 : sequence to ??
        2. 기계 번역(neural machine translation, NMT) : **인코더-디코더 구조를 만드는 RNN**
    - **Attention 메커니즘 활용** : 각 타임 스텝에서 모델이 집중해야 할 입력 부분을 선택하도록 학습하는 신경망 구조
        1. RNN 기반 인코더-디코더 구조의 성능을 높이는 방법
        2. **트랜스포머(Transformer) : RNN을 모두 제거하고 어텐션만 사용해 성능 확보**
    - 최근 발전된 모델 검토(트랜스포머 기반_GPT, BERT), 허깅 페이스의 트랜스포머 라이브러리 활용 방법

---
## **16.4 어텐션 메커니즘**
---

### **들어가면서** 

<hr style="border:0;border-top:2px dashed gray;">

**바흐다나우 Attention = 연결(concatenative) = 덧셈(additive) Attention**

- 논문 : Dzmitry Bahdanau et al., "Neural Machine Translation by Jointly Learning to Align and Translate", 2014
  - 저자들을 각 타임 스텝에서 (인코더에 의해 인코딩된 대로) **적절한 단어에 디코더가 초점을 맞추도록 하는 기술**을 소개 -> 입력 단어에서 번역까지 경로가 훨씬 짧아지기 때문에 RNN의 단기 기억 제한성이 훨씬 개선됨
  - 인코더의 모든 출력을 디코더로 전송 -> 인코더의 모든 출력을 다루기 위하여 **인코더 출력의 가중치 합을 계산하여 해당 단계에서 주의를 집중할 단어를 결정**
  - **α(t, i) : 가중치는 t번째 디코더 타임 스텝에서 i번째 인코더 출력의 가중치**

  <img src="./screenshot/16-6.png" width="600">

  **그림 16-7 : 어텐션 모델과 인코더-디코더 네트워크를 사용한 신경망 기계 번역**

- **작동 방식**
  - 타임 스텝마다 메모리 셀이 입력(인코더 출력의 가중치 합)과 이전 타입 스텝의 은닉상태(h)를 받음
  - (그림에는 나타나 있지는 않지만) 이전 타임 스텝에서 타깃 단어를 받음, 또는 추론 시에는 이전 타임 스텝의 출력(y')을 받음
  - α(t, i) 가중치는 **정렬 모델(또는 어텐션 층, attention layer)에서 생성되며, 해당 모델은 인코더-디코더 모델과 함께 훈련**

- **케라스는 바흐다나우 어텐션을 위해 AdditiveAttention 층을 제공**

<hr style="border:0;border-top:2px dashed gray;">

**루옹 Attention = 곱셈(multiplicative) Attention**

- 논문 : Minh-Thang Luong et al., "Effective Approachs to Attention-Based Neural Machine Translation", 2015
  - 이 정렬 모델의 목적은 **인코더의 출력 하나와 디코더의 이전 은닉 상태 사이의 유사도를 측정하는 것** -> 유사도를 벡터간 점곱 형식으로 제안 
  - 루옹 등이 제안한 간소화된 버전은
    1. **이전 타임 스텝이 아니라 현재 타임 스텝에서 디코더의 은닉 상태를 사용(h(t))**
    2. **어텐션 메커니즘의 출력(h(t)_tilde)을 사용하여 (디코더의 현재 은닉상태를 계산하지 않고) 바로 디코더의 예측을 계산**
    3. **인코더의 출력이 점곱 계산전에 먼저(편향이 없는) 완전 연결층을 통과하는 점곱 메커니즘의 변형을 제안 -> '일반' 점곱 방법** -> (추측) 훈련할 수 있는 가중치를 늘리는 효과


  <img src="./screenshot/16-7.png" width="300">

  **그림 16-7 : Attention Mechanism**

- **케라스는 루옹 어텐션을 위해 tf.kears.layers.Attention 층을 제공**
---

### **RNN 기반 인코더-디코더 구조의 성능을 높이는 방법**

<hr style="border:0;border-top:1.5px dashed gray;">

#### **데이터 준비 및 전처리**

In [ ]:
# 영어/스페인어 문장 쌍으로 구성된 데이터셋 다운로드

url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip" #URL 변수에 데이터셋이 저장된 zip 파일의 다운로드 링크 지정
path = keras.utils.get_file("spa-eng.zip", origin=url,                        #해당 URL에서 지정한 zip 파일을 다운로드하고,
                            cache_dir="datasets", extract=True)                #'datasets" 폴더에 저장, 압축을 해제, 반환값 'path'는 압축파일의 전체 경로
#text = (Path(path).with_name("spa-eng") / "spa.txt").read_text()            #압축 해제된 폴더에서 'spa.txt' 파일을 읽어와서 'text' 변수에 저장
text = (Path(path).parent / "spa-eng_extracted" / "spa-eng"/"spa.txt").read_text()

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [6]:
# TextVectorization 층이 처리하지 못하는 특수문자 제거 -> 문자 쌍을 파싱 -> 셔플링
text = text.replace("¡", "").replace("¿", "")                               #텍스트에서 특수문자 제거
pairs = [line.split("\t") for line in text.splitlines()]                    #텍스트를 줄 단위로 분리하고, 각 줄을 탭 문자('\t')로 분리하여 문장 쌍을 생성 -> [영어문장, 스페인어문장] 형태의 리스트
np.random.shuffle(pairs)                                                    #문장 쌍을 무작위로 섞음
sentences_en, sentences_es = zip(*pairs)                                    #문장 쌍을 영어 문장과 스페인어 문장으로 분리하여 각각의 튜플로 저장
print(f"전체 문장 쌍 개수: {len(sentences_en)}")

# 처음 세 개의 문장 쌍을 확인
for i in range(3):
    print(f"영어: {sentences_en[i]}\n스페인어: {sentences_es[i]}\n")

전체 문장 쌍 개수: 118964
영어: How boring!
스페인어: Qué aburrimiento!

영어: I love sports.
스페인어: Adoro el deporte.

영어: Would you like to swap jobs?
스페인어: Te gustaría que intercambiemos los trabajos?



> **위 코드 설명**
1. *pairs는 리스트의 각 요소(여기서는 [영어, 스페인어] 쌍)를 zip 함수에 각각의 인자로 전달합니다.
2. zip은 같은 위치끼리 묶어서, 첫 번째 값들끼리(모든 영어 문장) 두 번째 값들끼리(모든 스페인어 문장)를 각각 튜플로 만들어줍니다.

- 정리 : 
zip(*pairs)는
[ [A1, B1], [A2, B2], [A3, B3] ] → (A1, A2, A3), (B1, B2, B3)
처럼 "행" 중심의 리스트를 "열" 중심의 튜플로 바꿔주는 역할

##### **훈련 세트와 검증 세트 만들기**

> **입력 세트 만들기** 

In [7]:
# 처음 10만개 문장 쌍은 훈련에 사용하고 나머지는 검증에 사용
# 디코더 입력은 스페인어 문장과 시작 토큰으로 구성

X_train = tf.constant(sentences_en[:100_000])  # 처음 10만 개 영어 문장을 훈련 입력으로 사용
X_valid = tf.constant(sentences_en[100_000:])  # 나머지 영어 문장을 검증 입력으로 사용

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])  # 훈련용 디코더 입력: 시작 토큰 + 스페인어 문장
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])  # 검증용 디코더 입력: 시작 토큰 + 스페인어 문장

> **출력 세트 만들기**

In [ ]:
# 각 언어마다 하나씩 두 개의 TextVectorization 층을 생성하여 적용
vocab_size = 1000                                                           #어휘 사전 크기 지정, 최첨단 번역 모델은 30,000개 정도 사용
max_length = 50                                                             #최대 문장 길이 지정, 데이터셋에 있는 모든 문장은 최대 50단어
                                                                            #입력 시퀀스 길이가 모두 50토큰이 될 때까지 자동으로 0으로 패딩

text_vec_layer_en = keras.layers.TextVectorization(
    max_tokens=vocab_size, output_sequence_length=max_length)
text_vec_layer_es = keras.layers.TextVectorization(
    max_tokens=vocab_size, output_sequence_length=max_length)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es]) #스페인어 문장에 시작과 끝 토큰 추가(SOS, EOS), 실제 스페인어 단어가 아니기만 하면 다른 단어를 사용할 수 있음


2026-03-09 22:10:34.950382: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 18016872 exceeds 10% of free system memory.
2026-03-09 22:10:35.628334: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 15386752 exceeds 10% of free system memory.
2026-03-09 22:10:35.628739: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 23080128 exceeds 10% of free system memory.


In [9]:
# 두 어휘 사전에 있는 처음 10개의 토큰을 확인: 실제 단어는 빈도가 많은 순서대로 나열됨
text_vec_layer_en.get_vocabulary()[:10], text_vec_layer_es.get_vocabulary()[:10]

(['',
  '[UNK]',
  np.str_('the'),
  np.str_('i'),
  np.str_('to'),
  np.str_('you'),
  np.str_('tom'),
  np.str_('a'),
  np.str_('is'),
  np.str_('he')],
 ['',
  '[UNK]',
  np.str_('startofseq'),
  np.str_('endofseq'),
  np.str_('de'),
  np.str_('que'),
  np.str_('a'),
  np.str_('no'),
  np.str_('tom'),
  np.str_('la')])

In [10]:
# 타깃은 스페인어 문장과 끝 토큰으로 구성
y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])  # 훈련용 타깃: 스페인어 문장 + 종료 토큰을 정수 시퀀스로 변환
y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])  # 검증용 타깃: 스페인어 문장 + 종료 토큰을 정수 시퀀스로 변환

2026-03-09 22:10:51.730277: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 17012784 exceeds 10% of free system memory.
2026-03-09 22:10:51.832323: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 40000000 exceeds 10% of free system memory.


- 결과적으로 인코더에는 영어 문장이 입력되고, 디코더에는 "startofseq + 스페인어 문장"이 입력, 타깃에는 "스페인어 문장 + endofseq"가 들어가 번역 모델을 효과적으로 학습 시킬 수 있음

In [11]:
# 데이터 확인
print("문장 확인")
print(sentences_en[0])                                                      #처음 영어 문장 출력
print(sentences_es[0])                                                      #처음 스페인어 문장 출력
print("\n")
print("입력 텐서 확인")
print(X_train[0])                                                           #처음 영어 문장을 텐서화한 결과 출력
print(X_train_dec[0])                                                       #처음 디코더 입력 문장 출력
print(X_train.shape, X_train_dec.shape)                                     #훈련 입력과 디코더 입력의 텐서 형태 출력
print("\n")
print("타깃 텐서 확인")
print(y_train[0])                                                           #처음 디코더 출력 문장 출력(정수 시퀀스 형태)
print(y_train.shape)                                                        #훈련 타깃의 텐서 형태 출력

문장 확인
How boring!
Qué aburrimiento!


입력 텐서 확인
tf.Tensor(b'How boring!', shape=(), dtype=string)
tf.Tensor(b'startofseq Qu\xc3\xa9 aburrimiento!', shape=(), dtype=string)
(100000,) (100000,)


타깃 텐서 확인
tf.Tensor(
[25  1  3  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0], shape=(50,), dtype=int64)
(100000, 50)


> 출력은 “문자열 텐서 1개”를 보여줍니다.

- `tf.Tensor`: 텐서플로 텐서 객체입니다.  
- `shape=()` : 스칼라(0차원) 텐서, 즉 값 하나만 가진 텐서입니다.  
- `dtype=string` : 데이터 타입이 문자열입니다.  
- `numpy=b'...'` : 내부 값이 **바이트 문자열**로 저장돼 있다는 뜻입니다.  
  - `b'...'`는 파이썬의 bytes 표기이며, 내용은 `문자열`와 동일합니다.

<hr style="border:0;border-top:1.5px dashed gray;">

#### **모델 구축**
- 모델 구성이 순차적이지 않기 때문에 함수형 API를 사용

In [ ]:
# 임베딩 구성

# 인코더와 디코더에 각각 하나씩 두 개의 텍스트 입력이 필요
encoder_inputs_en = keras.Input(shape=[], dtype=tf.string, name="inputs_en")        #영어 입력
decoder_inputs_es = keras.Input(shape=[], dtype=tf.string, name="inputs_es")        #스페인어 입력

# TextVectorization 층을 사용하여 문장을 인코딩 -> 자동 마스킹 처리
encoder_input_ids = text_vec_layer_en(encoder_inputs_en)                    #영어 입력을 정수 시퀀스로 변환
decoder_input_ids = text_vec_layer_es(decoder_inputs_es)                    #스페인어 입력을 정수 시퀀스로 변환

# 임베딩 층을 사용하여 정수 시퀀스를 임베딩 벡터로 변환
embed_size = 128                                                            #임베딩 차원, 128차원으로 설정
encoder_embedding_layer = keras.layers.Embedding(input_dim=vocab_size,      #어휘 사전 크기
                                                 output_dim=embed_size,     #임베딩 차원
                                                 mask_zero=True,            #패딩 토큰(0)을 마스킹
                                                 name="encoder_embedding")
decoder_embedding_layer = keras.layers.Embedding(input_dim=vocab_size,
                                                 output_dim=embed_size,
                                                 mask_zero=True,            
                                                 name="decoder_embedding")
encoder_embeddings = encoder_embedding_layer(encoder_input_ids)             #영어 입력을 임베딩 벡터로 변환
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)             #스페인어 입력을 임베딩 벡터로 변환

<hr style="border:0;border-top:1.5px dashed gray;">

In [ ]:
# 인코더를 만들고 임베딩된 입력을 전달
# 인코더의 모든 출력을 Attention 층으로 전달해야 하므로 return_sequences=True로 설정
# 나중에 여유되면 양방향 LSTM 도 시도해보기
encoder = keras.layers.LSTM(512, return_state=True,                         #512차원 LSTM 인코더 층 생성, LSTM 층만 사용했지만 여러 개의 층을 사용할 수도 있음
                            return_sequences=True, name="encoder_lstm")    #return_state=True로 설정하여 인코더의 상태 벡터를 반환하도록 함
                                                                            
encoder_outputs, state_h, state_c = encoder(encoder_embeddings)             #임베딩된 영어 입력을 인코더에 전달하여 출력과 상태 벡터를 얻음(단기 상태와 장기 상태)
        # or -> encoder_outputs, *encoder_states = encoder(encoder_embeddings)      #LSTM 인코더는 상태 벡터를 반환하므로, * 연산자를 사용하여 unpacking도 가능

In [ ]:
# 디코더를 만들고 임베딩된 입력과 인코더 상태를 전달
decoder = keras.layers.LSTM(512, return_sequences=True, name="decoder_lstm")  #512차원 LSTM 디코더 층 생성, return_sequences=True로 설정하여 디코더의 출력 시퀀스를 반환하도록 함
decoder_outputs = decoder(decoder_embeddings, initial_state=[state_h, state_c])  #임베딩된 스페인어 입력과 인코더 상태를 디코더에 전달하여 출력 시퀀스를 얻음
        # or -> decoder_outputs = decoder(decoder_embeddings, initial_state=encoder_states)

In [ ]:
# Attention 층을 생성하고 디코더의 상태와 인코더의 출력을 전달(중요)
# 각 스텝에서 디코더의 상태를 참조하려면 사용자 정의 메모리 셀을 작성해야하므로 간단하게 하기 위해 디코더의 상태 대신 출력을 사용
attention_layer = keras.layers.Attention(name="attention_layer")            #Attention 층 생성

attention_outputs = attention_layer([decoder_outputs, encoder_outputs])     #디코더/인코더 출력(마스크제거)을 Attention 층에 전달하여 컨텍스트 벡터를 계산

- 케라스의 Attention과 AdditiveAttention 층은 **쿼리, 키 및 (선택적으로) 값에 해당하는 항목을 담은 리스트를 입력**으로 기대
- 값을 전달하지 않으면 자동으로 키를 값으로 사용
- 본 코드에서는 디코더 출력을 쿼리로, 인코더 출력을 키와 값으로 사용
- **각 디코터 출력(각 쿼리)에 대해 어텐션 층은 디코더 출력과 가장 유사한 인코더 출력(키/값)의 가중치 합을 반환**
- 결론적으로 **어텐션 메커니즘은 훈련 가능한 메모리 검색기** 역할을 하여, **디코더가 번역할 때마다 인코더의 모든 출력에서 가장 관련성 높은 부분을 선택할 수 있도록 함**

In [ ]:
# Attention 층의 출력을 소프트맥스 활성화 함수가 있는 Dense 층에 전달하여 각 스텝에 대한 단어 확률을 계산
output_layer = keras.layers.Dense(vocab_size, activation='softmax',         #어휘 사전 크기와 소프트맥스 활성화 함수를 사용하여 출력층 생성
                                  name="output_layer") 
Y_proba = output_layer(attention_outputs)                                   #Attention 층의 출력을 출력층에 전달하여 단어 확률을 계산 

<hr style="border:0;border-top:1.5px dashed gray;">

#### **모델 훈련**
- 케라스 Model 객체를 만들고 컴파일하고 훈련 

In [ ]:
model = keras.Model(inputs=[encoder_inputs_en, decoder_inputs_es],          #케라스 Model 객체 생성, 인코더 입력과 디코더 입력을 입력으로 사용하고,
                    outputs=[Y_proba])                                      #디코더의 출력 확률을 출력으로 사용
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="nadam", metrics=["accuracy"])                     #모델 컴파일, 손실 함수는 희소 범주형 교차 엔트로피, 최적화 알고리즘은 Nadam, 평가 지표는 정확도
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs_en           │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inputs_es           │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 50)        │          0 │ inputs_en[0][0]   │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization… │ (None, 50)        │          0 │ inputs_es[0][0]   │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 50, 128)   │    128,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 50)        │          0 │ text_vectorizati… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 50, 128)   │    128,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 50, 512), │  1,312,768 │ encoder_embeddin… │
│                     │ (None, 512),      │            │ not_equal[0][0]   │
│                     │ (None, 512)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ (None, 50, 512)   │  1,312,768 │ decoder_embeddin… │
│                     │                   │            │ encoder_lstm[0][… │
│                     │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 50, 512)   │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output_layer        │ (None, 50, 1000)  │    513,000 │ attention_layer[… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,394,536 (12.95 MB)

 Trainable params: 3,394,536 (12.95 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# 모델 훈련
history = model.fit(
    #입력 이름이 모델 정의 시 사용한 Input 레이어의 이름과 일치해야함을 주의
    x={"inputs_en": X_train, "inputs_es": X_train_dec}, y=y_train,          #영어 입력과 스페인어 디코더 입력을 딕셔너리 형태로 전달, 훈련용 타깃을 전달
    validation_data=({"inputs_en": X_valid, "inputs_es": X_valid_dec}, y_valid),  #검증 데이터도 딕셔너리 형태로 전달
    epochs=1,                                                              #에폭 수 설정
    batch_size=64                                                           #배치 크기 설정
)

# 간소화 버전 : model.fit((X_train, X_train_dec), y_train, validation_data=((X_valid, X_valid_dec), y_valid), epochs=10, batch_size=64)


2026-02-14 23:44:06.994459: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91900


1563/1563 ━━━━━━━━━━━━━━━━━━━━ 66s 40ms/step - accuracy: 0.3332 - loss: 3.7772 - val_accuracy: 0.4046 - val_loss: 3.0810


<hr style="border:0;border-top:1.5px dashed gray;">

#### **추론**
- 추론시 `model.predict()`를 호출하는 것처럼 간단하지 않음
- 디코더는 이전 타임 스텝에서 예측한 단어를 입력으로 기대하기 때문임
- 이를 수행하는 한가지 방법은 **이전 출력을 기록하여 다음 타임 스텝에서 인코더에게 주입하는 사용자 정의 메모리 셀을 만드는 것**
- **간단하게 구성하려면 모델을 여러번 호출하고 매번 단어를 하나씩 늘려서 예측하면 됨 -> 아래 간단한 유틸함수 예시**

In [17]:
# 모델을 여러번 호출하고 매번 단어를 하나씩 늘려서 예측하는 간단한 유틸함수
# 이 방식은 디코더가 한 번에 한 단어씩 예측하는 오토리그레시브(autoregressive) 번역의 전형적인 예
# 한번에 한 단어씩 계속 예측하여 점차적으로 번역을 완료하고 EOS 토큰에 도달하면 중지
def translate(sentence_en, model):
    model = model
    translation = ""
    for word_idx in range(max_length):                                      #반복문은 최대 max_lenth만큼 실행
        X = tf.constant([sentence_en])                                      #입력 문장을 텐서로 변환
        X_dec = tf.constant(["startofseq " + translation])                  #현재까지 번역된 문장에 시작 토큰을 추가
        y_proba = model.predict({"inputs_en": X, "inputs_es": X_dec}, verbose=0)[0, word_idx]
        # predict() 함수는 전체 시퀀스에 대한 예측 결과(확률분포 배열)를 반환
        # 반환된 결과는 일반적으로 (배치 크기, 시퀀스 길이, 어휘 크기) 형태의 3차원 배열
        # [0, word_idx]는 첫번째 샘플(배치)에서 word_idx 위치(현재 예측할 단어)에 해당하는 확률 분포(어휘 크기만큼의 벡터)를 선택
        # 이 벡터는 해당 위치에서 각 단어가 나올 확률을 나타냄
        predicted_word_id = np.argmax(y_proba)                              #가장 확률이 높은 단어 선택, 예측된 단어의 인덱스
        predicted_word = text_vec_layer_es.get_vocabulary()[predicted_word_id]  #예측된 단어의 인덱스를 사용하여 단어를 가져옴
        if str(predicted_word) == "endofseq":
            break
        translation += " " + str(predicted_word)                                 #예측된 단어를 번역에 추가
    return translation.strip()                                              #번역 결과를 반환, 앞뒤 공백 제거
# 주의해야할 부분은 디코더 입력에 항상 시작 토큰을 붙여야하며, 번역 결과에 단어를 누적해서 전달해야함 

In [18]:
# 추론: 번역 예시
print(translate("I like soccer", model))                                    #번역 성공             
print(translate("I like soccer and also going to the beach", model))        #번역 성공     

me me me [UNK]
gusta [UNK] a la fiesta a la fiesta


---
### **16.4.1 트랜스포머 구조: 어텐션만 있으면 된다**
- 논문 : Ashish Vaswani et al., "Attention is All You Need", 2017 -> Craft에 논문 스터디
    - 순환층이나 합성곱층을 전혀 사용하지 않고 (임베딩층, 밀집층, 정규화층, 몇 가지 다른 구성 요소와) 어텐션 메커니즘만 사용하여 NMT 분야의 SOTA를 달성
    - **장점:**
        - 순환하지 않기 때문에 RNN처럼 **그레디언트 소실이나 폭주 문제가 없음**
        - 더 적은 단계로 학습 가능, **여러 GPU에 걸쳐 병렬화하기 용이**
        - **RNN보다 장기 패턴을 더 잘 포착** 

    <img src="./screenshot/16-8.png" width="400">

    **그림 16-8 : 2017년의 원본 트랜스포머 구조**

    - **훈련과 추론**
        - (영어 to 스페인어) NMT **훈련시** : 영어 문장을 인코더에 주입하고 이에 해당하는 스페인어 번역(타겟)을 디코더에 공급(스페인어 각 문장의 시작 부분에 추가적인 SOS 토큰 삽입 필요)
        - (영어 to 스페인어) NMT **추론시** : **트랜스포머를 여러번 호출하여 한 번에 한 단어씩 번역을 생성(출력)하고 매번 부분 번역을 디코더에 공급**
    - **전반적인 구조**
        - **임베딩 층**은 크기가 [배치 크기, 시퀀스 크기, 임베딩 크기]인 3D 텐서를 출력 -> 텐서를 통과하면서 점차 변형되지만 크기는 동일하게 유지.
        - **인코더**의 역할은 각 단어의 표현이 문맥 안에서 단어의 의미를 완벽하게 포착할 때까지 입력을 점진적으로 변환하는 것.
        - **디코더**의 역할은 번역된 문장의 각 단어 표현을 다음 번역 단어의 표현으로 점진적으로 변환하는 것.
    - **시퀀스 처리가 가능하도록 하는 요소:**
        - 인코더의 **멀티 헤드 어텐션(self attention)** : 같은 문장에 있는 다른 모든 단어에 주의를 기울여 각 단어 표현을 업데이트
        - 디코더의 **마스크드 멀티 헤드 어텐션(masked self attention?)** : 인코더의 self attention 층과 동일한 작업을 수행하나, 한 단어를 처리할 때 그 뒤에 있는 단어에는 관심을 기울이지 않는 인과적(casual) 작용
        - **Cross attention** : 디코더에서 위쪽에 있는 멀티 헤드 어텐션 층은 인코더 출력에 주의를 기울임
        - **위치 인코딩(positional encoding)** : n번째 위치 인코딩이 각 문장에 있는 n번째 단어의 단어 임베딩에 더해짐, 이렇게 하는 이유는 트랜스포머 구조에 있는 모든 층이 단어 위치를 무시하기 때문임, ***단어의 순서가 중요하므로 어떤 식으로든 트랜스포머에 위치 정보를 제공해야 하므로 단어 표현에 위치 인코딩을 추가하는 것**, "The Multi-Head Attention layers do not consider the order or the position ofthe words; they only look at their relationships."
    - **Self Attention과 Cross Attention의 차이(중요)**
        - 그림 16-8에서 멀티 헤드 어텐션 층으로 향하는 **처음 두 화살표는 키와 값을 나타내고 세 번째 화살표는 쿼리**를 나타냄
        - **Self Attetion** 층에서는 쿼리(문의), 키(인덱스), 값이 **모두 이전 층에서 출력한 단어 표현과 동일**
        - **Cross Attention** 층에서는 **인코더(바꾸고 싶은 대상) 최종 단어 표현에서 키와 값이 오고**, **쿼리는 디코더(생성되는 대상)의 해당 어텐션 이전 층에서 출력한 단어 표현에서 옴**
            

<hr style="border:0;border-top:1.5px dashed gray;">

#### **데이터 준비 및 전처리**
- "RNN 기반 인코더-디코더 구조의 성능을 높이~" 부분 참고

In [4]:
# 영어/스페인어 문장 쌍으로 구성된 데이터셋 다운로드

url = "https://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip" #URL 변수에 데이터셋이 저장된 zip 파일의 다운로드 링크 지정
path = keras.utils.get_file("spa-eng.zip", origin=url,                        #해당 URL에서 지정한 zip 파일을 다운로드하고,
                            cache_dir="datasets", extract=True)                #'datasets" 폴더에 저장, 압축을 해제, 반환값 'path'는 압축파일의 전체 경로
#text = (Path(path).with_name("spa-eng") / "spa.txt").read_text()            #압축 해제된 폴더에서 'spa.txt' 파일을 읽어와서 'text' 변수에 저장
text = (Path(path).parent / "spa-eng_extracted" / "spa-eng"/"spa.txt").read_text()

2638744/2638744 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [5]:
# TextVectorization 층이 처리하지 못하는 특수문자 제거 -> 문자 쌍을 파싱 -> 셔플링
text = text.replace("¡", "").replace("¿", "")                               #텍스트에서 특수문자 제거
pairs = [line.split("\t") for line in text.splitlines()]                    #텍스트를 줄 단위로 분리하고, 각 줄을 탭 문자('\t')로 분리하여 문장 쌍을 생성 -> [영어문장, 스페인어문장] 형태의 리스트
np.random.shuffle(pairs)                                                    #문장 쌍을 무작위로 섞음
sentences_en, sentences_es = zip(*pairs)                                    #문장 쌍을 영어 문장과 스페인어 문장으로 분리하여 각각의 튜플로 저장
print(f"전체 문장 쌍 개수: {len(sentences_en)}")

# 처음 세 개의 문장 쌍을 확인
for i in range(3):
    print(f"영어: {sentences_en[i]}\n스페인어: {sentences_es[i]}\n")

전체 문장 쌍 개수: 118964
영어: How boring!
스페인어: Qué aburrimiento!

영어: I love sports.
스페인어: Adoro el deporte.

영어: Would you like to swap jobs?
스페인어: Te gustaría que intercambiemos los trabajos?



##### **훈련 세트와 검증 세트 만들기**

> **입력 세트 만들기** 

In [6]:
# 처음 10만개 문장 쌍은 훈련에 사용하고 나머지는 검증에 사용
# 디코더 입력은 스페인어 문장과 시작 토큰으로 구성

X_train = tf.constant(sentences_en[:100_000])  # 처음 10만 개 영어 문장을 훈련 입력으로 사용
X_valid = tf.constant(sentences_en[100_000:])  # 나머지 영어 문장을 검증 입력으로 사용

X_train_dec = tf.constant([f"startofseq {s}" for s in sentences_es[:100_000]])  # 훈련용 디코더 입력: 시작 토큰 + 스페인어 문장
X_valid_dec = tf.constant([f"startofseq {s}" for s in sentences_es[100_000:]])  # 검증용 디코더 입력: 시작 토큰 + 스페인어 문장

> **출력 세트 만들기**

In [7]:
# 각 언어마다 하나씩 두 개의 TextVectorization 층을 생성하여 적용
vocab_size = 1000                                                           #어휘 사전 크기 지정, 최첨단 번역 모델은 30,000개 정도 사용
max_length = 50                                                             #최대 문장 길이 지정, 데이터셋에 있는 모든 문장은 최대 50단어
                                                                            #입력 시퀀스 길이가 모두 50토큰이 될 때까지 자동으로 0으로 패딩

text_vec_layer_en = keras.layers.TextVectorization(
    max_tokens=vocab_size, output_sequence_length=max_length)
text_vec_layer_es = keras.layers.TextVectorization(
    max_tokens=vocab_size, output_sequence_length=max_length)

text_vec_layer_en.adapt(sentences_en)
text_vec_layer_es.adapt([f"startofseq {s} endofseq" for s in sentences_es]) #스페인어 문장에 시작과 끝 토큰 추가(SOS, EOS), 실제 스페인어 단어가 아니기만 하면 다른 단어를 사용할 수 있음


2026-03-15 22:10:16.186459: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 18016872 exceeds 10% of free system memory.
2026-03-15 22:10:16.743626: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 23080128 exceeds 10% of free system memory.


In [9]:
# 타깃은 스페인어 문장과 끝 토큰으로 구성
y_train = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[:100_000]])  # 훈련용 타깃: 스페인어 문장 + 종료 토큰을 정수 시퀀스로 변환
y_valid = text_vec_layer_es([f"{s} endofseq" for s in sentences_es[100_000:]])  # 검증용 타깃: 스페인어 문장 + 종료 토큰을 정수 시퀀스로 변환

2026-03-15 22:10:26.307765: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 17012784 exceeds 10% of free system memory.
2026-03-15 22:10:26.365340: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 40000000 exceeds 10% of free system memory.


<hr style="border:0;border-top:1.5px dashed gray;">

#### **임베딩 구성**

In [12]:
# 인코더와 디코더에 각각 하나씩 두 개의 텍스트 입력이 필요
encoder_inputs_en = keras.Input(shape=[], dtype=tf.string, name="inputs_en")        #영어 입력
decoder_inputs_es = keras.Input(shape=[], dtype=tf.string, name="inputs_es")        #스페인어 입력

# TextVectorization 층을 사용하여 문장을 인코딩 -> 자동 마스킹 처리
encoder_input_ids = text_vec_layer_en(encoder_inputs_en)                    #영어 입력을 정수 시퀀스로 변환
decoder_input_ids = text_vec_layer_es(decoder_inputs_es)                    #스페인어 입력을 정수 시퀀스로 변환

# 임베딩 층을 사용하여 정수 시퀀스를 임베딩 벡터로 변환
embed_size = 128                                                            #임베딩 차원, 128차원으로 설정
encoder_embedding_layer = keras.layers.Embedding(input_dim=vocab_size,      #어휘 사전 크기
                                                 output_dim=embed_size,     #임베딩 차원
                                                 mask_zero=True,            #패딩 토큰(0)을 마스킹
                                                 name="encoder_embedding")
decoder_embedding_layer = keras.layers.Embedding(input_dim=vocab_size,
                                                 output_dim=embed_size,
                                                 mask_zero=True,            
                                                 name="decoder_embedding")
encoder_embeddings = encoder_embedding_layer(encoder_input_ids)             #영어 입력을 임베딩 벡터로 변환
decoder_embeddings = decoder_embedding_layer(decoder_input_ids)             #스페인어 입력을 임베딩 벡터로 변환

<hr style="border:0;border-top:1.5px dashed gray;">

#### **위치 인코딩**
- 문장 안에 있는 단어의 위치를 인코딩한 밀집 벡터, 단순히 **i번째 위치 인코딩이 문장에 있는 i번째 단어의 단어 임베딩에 더해짐**
- **구현 방법 1:** 
    - **Embedding층을 사용**하여 0부터 **배치에 있는 최대 시퀀스 길이까지** 모든 위치를 인코딩한 다음 그 결과를 단어 임베딩에 추가
    - 브로드캐스팅 규칙을 사용하면 모든 입력 시퀀스에 위치 인코딩이 적용
    - 이 구현은 임베딩이 래그드 텐서가 아닌 일반 텐서로 표현된다고 가정(최신 텐서플로 버전에서는 래그드 텐서 사용 가능

In [13]:
# 위 구현방법 1: Embedding 층을 사용하여 위치 인코딩 예시
max_length = 50                                                             #최대 문장 길이 지정, 데이터셋에 있는 모든 문장은 최대 50단어
embed_size = 128                                                            #임베딩 차원, 128차원으로 설정

pos_embed_layer = keras.layers.Embedding(max_length, embed_size)            #최대 길이와 임베딩 차원을 사용하여 위치 임베딩 층 생성
# 인코더와 디코더는 임베딩 크기가 동일하기 때문에 위치 인코딩을 위해 하나의 Embedding 층을 공유

# KerasTensor에는 keras.ops 사용
batch_max_len_enc = keras.ops.shape(encoder_embeddings)[1]                   #배치에 있는 시퀀스의 최대 길이 계산
encoder_in = encoder_embeddings + pos_embed_layer(keras.ops.arange(batch_max_len_enc))  #배치에 있는 모든 시퀀스에 위치 인코딩 적용

batch_max_len_dec = keras.ops.shape(decoder_embeddings)[1]                   #배치에 있는 시퀀스의 최대 길이 계산
decoder_in = decoder_embeddings + pos_embed_layer(keras.ops.arange(batch_max_len_dec))  #배치에 있는 모든 시퀀스에 위치 인코딩 적용

- **구현 방법 2:**
    - "Attention is All you Need" 저자들은 훈련 가능한 임베딩 대신 **여러가지 주기의 사인과 코사인 함수에 기반한 고정된 위치 인코딩을 선택**
    - 해당 방법은 학습된 위치 임베딩과 동일한 성능을 내면서 **모델에 파라미터를 추가하지 않고 임의의 긴 문장으로 확장 가능**
    - **사전 훈련 데이터가 많은 경우 일반적으로 훈련 가능한 위치 인코딩이 선호됨**
    
    <img src="./screenshot/16-9.png" width="600">

    **그림 16-9 : 위치 인코딩 식과 (전치된) 사인/코사인 위치 인코딩 행렬(위), 두 i 값에 대한 그래프(아래쪽)**
    : Sine/cosine positional embedding matrix (transposed, top) with a focus on two values of i (bottom)

    - 그림 해석 1: 22번째 위치에 있는 단어의 위치 인코딩은 [그림 16-9]의 왼쪽 위 수직 점선으로 나타나고, **이 위치에 해당하는 인코딩은 고유**
    - 그림 해석 2: The choice of oscillating functions (sine and cosine) **makes it possible for the model to learn relative positions as well.**
    - 그림 해석 3: Words located 38 words apart (e.g., at positions p = 22 and p = 60) always have the same positional embedding values in the embedding dimensions i = 100 and i = 101. This explains why we need both the sine and the cosine for each frequency: if we only used the sine (the blue wave at i = 100), the model would not be able to distinguish positions p = 25 and p = 35 (marked by a cross).

In [14]:
# 위 구현 방법 2: 사인과 코사인 함수를 사용하여 위치 인코딩
# 원 트랜스포머 논문에서 제안된 사인/코사인 기반의 고정 위치 인코딩을 케라스 커스텀 레이어로 구현
'''
1. 생성자에서 위치 인코딩 행렬을 미리 계산(효율성 향상)
2. call() 메서드에서 이 인코딩 행렬을 입력의 최대 길이로 잘라 입력에 더함
3. 입력 마스크를 다음 층으로 전파하기 위해 supports_masking=True로 설정
'''
class PositionalEncoding(keras.layers.Layer):                               #위치 인코딩을 위한 커스텀 클래스 정의
    def __init__(self, max_length, embed_size, dtype=tf.float32, **kwargs): #최대 시퀀스 길이, 임베딩 차원, 데이터 타입을 초기화 인자로 받음
        super().__init__(dtype=dtype, **kwargs)                             #부모 클래스의 생성자 호출(초기화), 데이터 타입과 기타 인자를 전달
        assert embed_size % 2 == 0, "embed size must be even"               #임베딩 차원이 짝수여야함 (사인/코사인 쌍으로 나누기 위해)
        
        # 최대 길이와 임베딩 차원에 대한 그리드 생성
        p, i = np.meshgrid(np.arange(max_length),                           #p: 위치 인덱스(0부터 max_length-1까지), 0-49
                           2*np.arange(embed_size//2))                      #i: 임베딩 차원 인덱스(0부터 embed_size/2-1까지, 짝수 인덱스만), [0,2,4,...,126]
        
        # 위치 인코딩 행렬 계산, 짝수 인덱스에는 사인 함수, 홀수 인덱스에는 코사인 함수 적용
        # 짝수 차원: ( \sin(p / 10000^{i/d}) ), 홀수 차원: ( \cos(p / 10000^{i/d}) )
        pos_emb = np.empty((1, max_length, embed_size))                     #위치 인코딩 행렬을 저장할 배열 생성, (배치차원_1, 위치_50, 임베딩차원_128) 크기의 빈 행렬 생성
        pos_emb[0, :, ::2] =np.sin(p/10_000 ** (i/embed_size)).T            #짝수 인덱스(0,2,4...)에 사인값을 계산해 저장
        pos_emb[0, :, 1::2] = np.cos(p/10_000 ** (i/embed_size)).T          #홀수 인덱스(1,3,5...)에 코사인값을 계산해 저장
        self.pos_encodings = tf.constant(pos_emb.astype(self.dtype))        #계산된 위치 인코딩 행렬을 텐서플로 상수로 변환하여 클래스 속성으로 저장
        self.supports_masking = True                                        #이 층이 마스킹을 지원함(패딩 등)을 나타내는 속성 설정

    def call(self, inputs):                                                 #레이어가 호출될 때 실행되는 메서드 정의
        batch_max_length = tf.shape(inputs)[1]                              #입력의 실제 시퀀스 길이(두번째 차원)을 추출
        return inputs + self.pos_encodings[:, :batch_max_length]            #입력에 위치 인코딩을 더하여 반환, 위치 인코딩은 입력 시퀀스 길이에 맞게 잘라서(슬라이싱) 사용

In [15]:
# 인코더와 디코더에 위치 인코딩 적용
pos_embed_layer = PositionalEncoding(max_length, embed_size)                #위치 인코딩 레이어 생성
encoder_in = pos_embed_layer(encoder_embeddings)                            #인코더 임베딩에 위치 인코딩 적용
decoder_in = pos_embed_layer(decoder_embeddings)                            #디코더 임베딩에 위치 인코딩 적용

<hr style="border:0;border-top:1.5px dashed gray;">

#### **인코더 구현**

> ##### **멀티 헤드 어텐션(Self Attention)**
- 스케일드 접곱 어텐션(scaled dot-product attention)

<img src="./screenshot/윤성로_교수님_수업자료.png" width="800">

- **그림 : 멀티 헤드 어텐션 설명(윤성로 교수님 수업자료) // Self-Attention의 경우임**
- **Cross Attention의 경우에는 인코더 최종 단어 표현에서 Key와 Value가 오고, 쿼리는 디코더(생성되는 대상)의 해당 어텐션 이전 층에서 출력한 단어 표현에서 옴**

    - **Q** 는 행마다 쿼리를 하나를 담은 행렬, 크기는 **[n_queries, d_keys]**, n_queries는 쿼리 개수, d_keys는 쿼리와 키의 차원 개수(보통은 embedding 차원보다는 작게 설정)
    - **K** 는 행마다 키 하나를 담은 행렬 **[n_keys, d_keys]**, **V** 는 행마다 값 하나를 담은 행렬 **[n_keys, d_values]**
- **(QK^T)의 크기는 [n_queries, n_keys], 이 행렬은 쿼리/키 쌍마다 하나의 유사도 점수를 담고 있음**
    - 소프트맥스 함수의 출력도 동일한 크기이지만 모든 행의 합이 1임
    - **최종 출력의 크기는 [n_queries, d_values] 이며, 하나의 행은 쿼리 결과(값의 가중치 합)을** 나타냄
- 스케일링 인자 `1/(d_keys^0.5)` 는 소프트맥스 함수가 포화되어 gradient가 너무 작아지지 않도록 유사도 점수를 낮춤 -> **tf.keras.layers.Attention 층을 만들때 use_scale=True로 지정하면 유사도 점수의 스케일을 적절히 낮추는 방법을 배울 수 있는 추가 파라미터가 생성**
- 소프트맥스 함수를 계산하기 전에 유사도 점수에 아주 큰 음수를 더해 일부 키/값 쌍을 제외하도록 마스킹 처리할 수 있음 -> **Masked Multi-head Attention에서 사용하는 방식임**

> ##### **인코더 구조**
- 윤성로 교수님 수업 자료 중42쪽 그림 넣기

In [16]:
N = 2                                                                       #N은 6대신 2로 설정하여 훈련 시간을 단축
num_heads = 8                                                               #어텐션 헤드 수
dropout_rate = 0.1                                                          #드롭아웃 비율
n_units = 128                                                               #피드포워드 블록의 첫번째 밀집층
d_keys = embed_size // num_heads                                            #각 어텐션 헤드의 키 차원, 임베딩 차원을 어텐션 헤드 수로 나눈 값

#encoder_pad_mask = tf.math.not_equal(encoder_input_ids,0)[:, tf.newaxis]    #인코더 입력에서 패딩 토큰(0)이 아닌 위치를 True로 표시하는 마스크 생성, shape: (배치크기, 1, 최대시퀀스길이)
encoder_pad_mask = keras.ops.not_equal(encoder_input_ids, 0)                #KerasTensor에는 keras.ops 사용
encoder_pad_mask = keras.ops.expand_dims(encoder_pad_mask, axis=1)          #shape: (배치크기, 1, 최대시퀀스길이)
Z = encoder_in                                                              #인코더 입력에 위치 인코딩이 적용된 결과

for _ in range(N):
    skip = Z                                                                #스킵 연결을 위한 임시 변수
    attn_layer = keras.layers.MultiHeadAttention(                           #멀티헤드 어텐션 층 생성
        num_heads=num_heads, key_dim=d_keys, dropout=dropout_rate)      
    
    Z = attn_layer(query=Z, value=Z, attention_mask=encoder_pad_mask)       #자기 자신에 대한 어텐션 계산, attention_mask에서 [배치크기, 최대쿼리길이, 최대값길이] 크기의 불리언 텐서를 받음
    Z = keras.layers.LayerNormalization()(keras.layers.Add()([Z, skip]))    #스킵 연결과 레이어 정규화 적용

    skip = Z                                                                #스킵 연결을 위한 임시 변수
    Z = keras.layers.Dense(n_units, activation="relu")(Z)                   #첫번째 밀집층(피드포워드 블록)
    Z = keras.layers.Dense(embed_size)(Z)                                   #두번째 밀집층(피드포워드 블록)
    Z = keras.layers.Dropout(dropout_rate)(Z)                               #드롭아웃 적용
    Z = keras.layers.LayerNormalization()(keras.layers.Add()([Z, skip]))    #스킵 연결과 레이어 정규화 적용
    

<hr style="border:0;border-top:1.5px dashed gray;">

#### **디코더 구현**

> ##### **Cross Attention** 

In [ ]:
# Bidirectional 층을 번역 모델의 인코더로 사용
encoder_2 = keras.layers.Bidirectional(
    keras.layers.LSTM(512, return_state=True, name="enoder_bi_LSTM"))       #이제 두 개가 아니라 네 개의 상태 벡터를 반환
# 네 개의 상태를 디코더의 LSTM 층의 초기 상태로 직접 사용할 수 없음 -> LSTM은 두 개의 상태(단기, 장기)만 기대하기 때문
# 정방향 h: (batch, 512), 정방향 c: (batch, 512), 역방향 h: (batch, 512), 역방향 c: (batch, 512)

# 따라서, 상태 벡터를 적절히 처리하여 디코더에 전달해야 함(두 개의 단기 상태와 두 개의 장기 상태를 연결)
encoder_outputs_2, *encoder_state = encoder_2(encoder_embeddings)           #인코더의 출력과 상태 벡터를 얻음
encoder_states = [tf.concat(encoder_state[::2], axis=-1),                   #단기상태 (0 & 2), shape: (batch, 1024)
                  tf.concat(encoder_state[1::2], axis=-1)]                  #장기상태 (1 & 3), shape: (batch, 1024)

#(주의): LSTM의 상태 벡터가(h, c) 쌍으로 번갈아 저장된다는 점과, concat의 axis 인자가 -1로 설정되어 마지막 차원(임베딩 차원)을 따라 연결된다는 점을 기억해야 함

decoder_2 = keras.layers.LSTM(1024, return_sequences=True, 
                              name="decoder_bi_LSTM")                       #디코더의 LSTM 층을 1024차원으로 변경(인코더의 상태 벡터가 1024차원이므로 디코더도 동일한 차원으로 설정)
decoder_outputs_2 = decoder_2(decoder_embeddings, initial_state=encoder_states)  #디코더의 초기 상태로 인코더의 상태를 전달

# output
output_layer_2 = keras.layers.Dense(vocab_size, activation='softmax', name="output_layer_2")  #어휘 사전 크기와 소프트맥스 활성화 함수를 사용하여 출력층 생성
Y_proba_2 = output_layer_2(decoder_outputs_2)                               #디코더의 출력 시퀀스를 출력층에 전달하여 단어 확률을 계산

In [ ]:
model_2 = keras.Model(inputs=[encoder_inputs_en, decoder_inputs_es],        #케라스 Model 객체 생성, 인코더 입력과 디코더 입력을 입력으로 사용하고,
                      outputs=[Y_proba_2])                                   #디코더의 출력 확률을 출력으로 사용
model_2.compile(loss="sparse_categorical_crossentropy",
                optimizer="nadam", metrics=["accuracy"])                   #모델 컴파일, 손실 함수는 희소 범주형 교차 엔트로피, 최적화 알고리즘은 Nadam, 평가 지표는 정확도

model_2.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     


 inputs_en (InputLayer)         [(None,)]            0           []                               
                                                                                                  
 text_vectorization (TextVector  (None, 50)          0           ['inputs_en[0][0]']              
 ization)                                                                                         
                                                                                                  
 inputs_es (InputLayer)         [(None,)]            0           []                               
                                                                                                  
 encoder_embedding (Embedding)  (None, 50, 128)      128000      ['text_vectorization[0][0]']     
                                                                                                  
 text_vectorization_1 (TextVect  (None, 50)          0           ['inputs_es[0][0]']              
 orization

In [17]:
# 모델 훈련
history = model_2.fit(
    #입력 이름이 모델 정의 시 사용한 Input 레이어의 이름과 일치해야함을 주의
    x={"inputs_en": X_train, "inputs_es": X_train_dec}, y=y_train,          #영어 입력과 스페인어 디코더 입력을 딕셔너리 형태로 전달, 훈련용 타깃을 전달
    validation_data=({"inputs_en": X_valid, "inputs_es": X_valid_dec}, y_valid),  #검증 데이터도 딕셔너리 형태로 전달
    epochs=10,                                                              #에폭 수 설정
    batch_size=64                                                           #배치 크기 설정
)
# val_accuracy가 미세하게 증가

Epoch 1/10
1563/1563 [==============================] - 72s 39ms/step - loss: 0.4033 - accuracy: 0.4397 - val_loss: 0.2772 - val_accuracy: 0.5585
Epoch 2/10
1563/1563 [==============================] - 59s 38ms/step - loss: 0.2311 - accuracy: 0.6153 - val_loss: 0.2102 - val_accuracy: 0.6459
Epoch 3/10
1563/1563 [==============================] - 63s 40ms/step - loss: 0.1755 - accuracy: 0.6894 - val_loss: 0.1880 - val_accuracy: 0.6770
Epoch 4/10
1563/1563 [==============================] - 63s 40ms/step - loss: 0.1415 - accuracy: 0.7389 - val_loss: 0.1807 - val_accuracy: 0.6881
Epoch 5/10
1563/1563 [==============================] - 65s 42ms/step - loss: 0.1148 - accuracy: 0.7807 - val_loss: 0.1802 - val_accuracy: 0.6916
Epoch 6/10
1563/1563 [==============================] - 64s 41ms/step - loss: 0.0922 - accuracy: 0.8184 - val_loss: 0.1846 - val_accuracy: 0.6912
Epoch 7/10
1563/1563 [==============================] - 62s 40ms/step - loss: 0.0741 - accuracy: 0.8512 - val_loss: 0.1916 -

In [18]:
# 추론: 번역 예시
print(translate("I like soccer", model_2))                                  #잘 작동
print(translate("I like soccer and also going to the beach", model_2))      #문제 발생 

1/1 [==============================] - 0s 27ms/step
me gusta el fútbol
1/1 [==============================] - 0s 40ms/step
me gusta el [UNK] y a la playa


---
- **(질문) 양방향 LSTM을 적용한 결과 훈련 정확도는 상당히 개선되는 것으로 보이는데 val_accuracy는 여전히 낮은 것처럼 보이네. 실제 추론에서도 긴 문장은 여전히 잘 번역이 안되는 것 같고 말이야. 너의 생각은 어때**
---
아주 좋은 관찰입니다!  
양방향 LSTM을 적용하면 훈련 정확도(train accuracy)는 확실히 올라가는 경우가 많지만, 검증 정확도(val_accuracy)가 크게 개선되지 않거나, 긴 문장 번역이 여전히 어렵게 느껴지는 현상은 자연스러운 결과입니다. 그 이유는 다음과 같습니다.

1. **기본 Seq2Seq 구조의 한계**

- 양방향 LSTM을 써도, **Attention이 없는 기본 Seq2Seq 구조**에서는 인코더가 입력 문장의 모든 정보를 "하나의 벡터"에 압축해야 합니다.
- 문장이 짧을 때는 이 벡터가 충분히 정보를 담을 수 있지만, **문장이 길어질수록 정보 손실이 커집니다.**
- 그래서 긴 문장일수록 번역 품질이 급격히 떨어지고, 검증 정확도도 낮게 나올 수 있습니다.

2. **과적합 가능성**

- 양방향 LSTM은 파라미터 수가 많아져서 훈련 데이터에는 더 잘 맞출 수 있습니다(훈련 정확도↑).
- 하지만 모델이 복잡해질수록 검증 데이터(새로운 문장)에는 일반화가 잘 안 될 수 있습니다(검증 정확도↓).

3. **Attention의 필요성**

- 실제로 NMT(신경망 기계 번역) 분야에서는 **Attention 메커니즘**이 도입되면서 긴 문장 번역 품질이 획기적으로 개선되었습니다.
- Attention은 디코더가 인코더의 "모든 타임스텝의 출력"을 참고해서, 번역할 때마다 입력 문장의 중요한 부분에 집중할 수 있게 해줍니다.
- 즉, **Attention이 없는 구조에서는 긴 문장 번역이 잘 안 되는 것이 정상**입니다.

4. **실전에서는**

- 최신 번역 모델(트랜스포머 등)은 모두 Attention을 기본으로 사용합니다.
- 만약 더 좋은 성능을 원한다면, **Attention 레이어를 추가하거나 트랜스포머 구조로 넘어가는 것이 필요**합니다.

**정리:**  
양방향 LSTM만으로는 긴 문장 번역이나 검증 정확도 개선에 한계가 있습니다.  
이 문제를 해결하려면 반드시 Attention 메커니즘을 도입해야 합니다!  
(교재 뒷부분에서 다루는 어텐션/트랜스포머 챕터를 참고해 보세요.)

#### **16.3.2 빔서치**
- **빔서치는 번역 결과의 품질을 높이기 위해, 한번에 가장 가능성 높은 단어 하나만 선택하는 것이 아니라, 여러 개(beam_width) 후보 문장을 동시에 추적하며 최적의 번역을 찾는 탐색 기법**
- **k개의 가능성 있는 문장의 리스트를 유지하고 디코더 단계마다 이 문장의 단어를 하나씩 생성하여 가능성 있는 k개의 문장을 만듦, 파라미터 k : 빔 너비(beam width)**
- 이 방식은 단순히 가장 확률이 높은 단어만 선택하는 그리디 서치보다 더 다양한 번역 후보를 고려할 수 있어, 실제 번역 품질이 향상되는 경우가 많음
- 추가적인 훈련을 사용하지 않고 사용법을 개선함으로써 인코더-디코더 모델의 성능을 높일 수 있음
- (Tip) 텐서플로 에드온 라이브러리에는 빔 서치 등을 포함하여 어텐션과 함께, 인코더-디코더 모델을 구축할 수 있는 완전한 seq2seq API가 포함되어 있음
- 양방향 RNN 기법과 마찬가지로 긴 문장에서의 성능이 좋지 않으므로 다음장에서는 이를 해결할 수 있는 어텐션 메커니즘을 다루게 됨

In [20]:
# 빔 서치 간단한 구현 코드 예시
def beam_search(sentence_en, model, beam_width=3, verbose=False):
    """
    1. 상위 첫 단어 예측
    2. 상위 k 번역 각각에 대해 해당 번역에 추가할 수 있는 모든 가능한 단어의 조건부 확률을 평가 -> 확장 번역과 해당 확률이 후보 목록에 추가
    3. 확률이 가장 높은 상위 k 후보만 유지
    4. 모든 번역이 EOS 토큰으로 완료될 때까지 다음 타임스텝으로 반복 -> 최상위 번역이 반환(해당 EOS 토큰을 제거한 후).

    * 참고: 
    - p(S)가 문장 S의 확률이고 p(W|S)가 번역이 S로 시작한다는 가정 하에 단어 W의 조건부 확률인 경우, 
    - 문장 S' = concat(S, W)의 확률은 p(S') = p(S) * p(W|S), 확률이 너무 작아져 부동 소수점 정밀도 오류가 발생할 수 있는 위험을 피하기 위해, 
    - 로그 확률로 계산 -> log(a*b) = log(a) + log(b), 따라서 log(p(S')) = log(p(S)) + log(p(W|S)).
    
    Args:
    - sentence_en: 번역할 영어 문장
    - model: 훈련된 seq2seq 모델
    - beam_width: 빔 너비(beam width), k개의 가능성 있는 문장의 리스트를 유지
    - verbose: True이면 상위 첫 단어를 표시
    """
    model = model
    
    # 첫 번째 단어의 확률 분포를 계산
    X = np.array([sentence_en])                                             #입력 영어 문장을 넘파이 배열로 변환 (배치 차원 추가) 
                                                                            #-> 해당 문장을 문자열 배열이 아니라 문장 1개로 이루어진 배열로 변환, 즉 문장이 늘어나면 배치가 늘어나는 형태
    X_dec = np.array(["startofseq"])                                        #디코더 입력을 시작 토큰으로 초기화
    y_proba = model.predict([{"inputs_en": X, "inputs_es": X_dec}])[0, 0]   #첫 번째 디코더 타임스텝의 단어별 확률 분포 예측
    top_k = tf.math.top_k(y_proba, k=beam_width)                            #확률이 높은 상위 beam_width개의 단어와 인덱스 추출
                                                                            #top_k는 (values, indices) 형태의 텐서로, values는 확률 값, indices는 단어 인덱스
    
    # 상위 첫 단어의 로그 확률과 해당 단어를 후보 리스트(top_translations)에 저장
    top_translations = [(np.log(word_proba), 
                         text_vec_layer_es.get_vocabulary()[word_id])       #(로그 확률, 단어) 튜플로 저장
                         for word_proba, word_id in zip(top_k.values, top_k.indices)]
    
    # verbose 모드에서 상위 첫 단어를 표시합니다.(로그 확률과 단어 출력)
    if verbose:
        print("상위 첫 단어:", top_translations) 
    
    # 빔 서치 반복 (각 타임스텝마다 후보 문장 확장)
    for idx in range(1, max_length):                                        #1부터 최대 문장 길이(max_length)만큼 반복
                                                                            #시작이 1부터인 이유는, 첫번째 단어(0번째 타임스텝)는 이미 앞에서 예측해서 후보군(top_translations)에 저장했기 때문
        candidates = []                                                     #후보 문장 리스트 초기화, 각 후보는 (로그 확률, 번역) 형태의 튜플로 저장
        for log_proba, translation in top_translations:                     #현재 후보 문장별로 확장
            
            if translation.endswith("endofseq"):                            #종료 토큰이 있는 경우, 
                candidates.append((log_proba, translation))                 #현재 번역을 후보로 추가하고 다음 반복으로 넘어감
                continue                                                    #주의: "다음 타임스탭"이 아니라 현재 타임스탭에서 top_translations 리스트의 다음 후보로 넘어가는 것
            
            X = np.array([sentence_en])                                     #입력 영어 문장 (배치 차원)
            X_dec = np.array(["startofseq " + translation])                 #현재까지 번역된 문장에 시작 토큰을 붙여 디코더 입력 생성
            y_proba = model.predict({"inputs_en": X,                        
                                     "inputs_es": X_dec})[0, idx]           #idx번째 타임스텝의 단어별 확률 분포 예측
            
            # 모든 단어에 대해 로그 확률을 누적하여 새로운 후보로 추가
            for word_id, word_proba in enumerate(y_proba):                  #모든 단어에 대해
                word = text_vec_layer_es.get_vocabulary()[word_id]          #단어 인덱스를 실제 단어로 변환
                candidates.append((log_proba + np.log(word_proba),          #로그 확률 누적
                                   f"{translation} {word}"))                #번역 결과 확장
        
        #로그 확률 기준으로 내림차순 정렬 후 상위 beam_width개 후보만 유지
        top_translations = sorted(candidates, reverse=True)[:beam_width]    

        # verbose 모드에서 현재까지의 최상위 후보 번역을 출력합니다.
        if verbose:
            print("지금까지 최상의 번역:", top_translations)

        # 모든 후보가 종료 토큰으로 끝나면 반복 종료, 최상위 번역 반환
        if all([tr.endswith("endofseq") for _, tr in top_translations]):
            return top_translations[0][1].replace("endofseq", "").strip()   #종료 토큰 제거 후 반환
        
# 주의해야할 부분은 로그 확률을 누적해야 한다는 점과, 디코더 입력에 항상 현재까지의 번역 결과를 누적해서 넣어야 한다는 점

In [35]:
# 추론: 번역 예시
# 정답) "Me gustan los gatos y los perros" or "Me encatan los gatos y los perros"
sentence_en = "I love cats and dogs"
print(translate(sentence_en, model_2))                                      #문제 발생
print(beam_search(sentence_en, model_2, verbose=True, beam_width=3))        #문제 발생

1/1 [==============================] - 0s 21ms/step
[UNK] a los perros y a ellos les [UNK]
1/1 [==============================] - 0s 20ms/step
상위 첫 단어: [(-0.6851277, '[UNK]'), (-1.0525336, 'me'), (-3.1773915, 'no')]
1/1 [==============================] - 0s 23ms/step
지금까지 최상의 번역: [(-0.7973547, '[UNK] a'), (-1.0564748, 'me [UNK]'), (-3.540265, 'no [UNK]')]
1/1 [==============================] - 0s 21ms/step
지금까지 최상의 번역: [(-1.264612, '[UNK] a los'), (-1.9125161, 'me [UNK] y'), (-1.9897163, '[UNK] a ambos')]
1/1 [==============================] - 0s 23ms/step
지금까지 최상의 번역: [(-1.2710826, '[UNK] a los perros'), (-2.1384048, '[UNK] a ambos a'), (-2.6463542, 'me [UNK] y a')]
1/1 [==============================] - 0s 21ms/step
지금까지 최상의 번역: [(-1.3368676, '[UNK] a los perros y'), (-2.1744833, '[UNK] a ambos a los'), (-3.483333, 'me [UNK] y a los')]
1/1 [==============================] - 0s 29ms/step
지금까지 최상의 번역: [(-1.5253516, '[UNK] a los perros y a'), (-2.1787865, '[UNK] a ambos a los perros'), 